# 02 · Train turn-detection experiment (E1-E4)

**Kaggle settings:** GPU **T4 x2 or P100** · **Internet ON** (downloads
whisper-tiny once) · ~1-2 h per experiment.

**Attach as input datasets** (Add Input):
1. `smart-turn-enhi-prep` — output of notebook 01
2. `hinglish-synth` — the uploaded synthetic Hinglish dataset
3. *(only when resuming)* the previous version's output of THIS notebook

**Run an experiment:** set `EXPERIMENT` in the config cell to one of
`e1_baseline` · `e2_hinglish_aug` · `e3_tinymel_scratch` · `e4_no_pause_aug`,
then *Save Version → Save & Run All*. Repeat per experiment (one per session).

**Resume after a kill:** attach the previous run's output, set `RESUME_FROM`
to its path (e.g. `/kaggle/input/02-train/run_e2_hinglish_aug`), run again.
Training continues from the last checkpoint (≤500 steps lost).

**Afterwards:** download `run_<EXPERIMENT>/` (metrics.json, ckpt_best.pt,
model_fp32.onnx, model_int8.onnx) into the repo's `experiments/` folder.

The `turn_detector` package below is auto-generated from the tested repo
sources by `tools/build_notebooks.py` — edit the repo, not the cells.


In [ ]:
%pip install -q onnx onnxruntime onnxscript polars soundfile

In [ ]:
import os
os.makedirs("turn_detector", exist_ok=True)

In [ ]:
%%writefile turn_detector/__init__.py

In [ ]:
%%writefile turn_detector/common.py
"""Torch-free audio constants + windowing shared by training and inference."""

import numpy as np

SAMPLE_RATE = 16000
WINDOW_SECONDS = 8.0
N_SAMPLES = int(SAMPLE_RATE * WINDOW_SECONDS)   # 128000
N_FFT = 400
HOP = 160
N_MELS = 80
N_FRAMES = N_SAMPLES // HOP                      # 800 mel frames
N_ENCODER_POSITIONS = N_FRAMES // 2              # 400 after Whisper's stride-2 conv


def right_align(wav: np.ndarray, n_samples: int = N_SAMPLES) -> np.ndarray:
    """Keep the last n_samples; left-pad with zeros if shorter."""
    wav = wav[-n_samples:]
    if len(wav) < n_samples:
        wav = np.concatenate([np.zeros(n_samples - len(wav), dtype=wav.dtype), wav])
    return wav.astype(np.float32)

In [ ]:
%%writefile turn_detector/features.py
"""Whisper-compatible log-mel frontend, windowed to the LAST 8 seconds.

Reimplements transformers' WhisperFeatureExtractor math in torch (batchable,
GPU-capable) instead of numpy: STFT(n_fft=400, hop=160, hann, center/reflect),
drop last frame, slaney mel (80 bins, 0-8kHz), log10 -> clamp to max-8 ->
(x+4)/4. Parity with the HF extractor is asserted in tests/test_features.py.

Turn detection cares about how speech ENDS, so windows are right-aligned:
the final audio sample always lands at the last mel frame; short clips are
zero-padded on the LEFT.
"""

import torch
import torch.nn as nn

from turn_detector.common import (  # noqa: F401  (re-exported for callers)
    HOP, N_ENCODER_POSITIONS, N_FFT, N_FRAMES, N_MELS, N_SAMPLES,
    SAMPLE_RATE, WINDOW_SECONDS, right_align,
)


class LogMel(nn.Module):
    """waveform (B, 128000) float32 -> log-mel (B, 80, 800)."""

    def __init__(self):
        super().__init__()
        from transformers.audio_utils import mel_filter_bank
        filters = mel_filter_bank(
            num_frequency_bins=1 + N_FFT // 2,
            num_mel_filters=N_MELS,
            min_frequency=0.0,
            max_frequency=8000.0,
            sampling_rate=SAMPLE_RATE,
            norm="slaney",
            mel_scale="slaney",
        )  # (201, 80)
        self.register_buffer("mel_filters", torch.from_numpy(filters).float())
        self.register_buffer("window", torch.hann_window(N_FFT, periodic=True))

    @torch.no_grad()
    def forward(self, wav: torch.Tensor) -> torch.Tensor:
        if wav.dim() == 1:
            wav = wav.unsqueeze(0)
        stft = torch.stft(
            wav, N_FFT, HOP, window=self.window,
            center=True, pad_mode="reflect", return_complex=True,
        )
        magnitudes = stft[..., :-1].abs() ** 2                  # (B, 201, 800)
        mel = self.mel_filters.T @ magnitudes                   # (B, 80, 800)
        log_spec = torch.clamp(mel, min=1e-10).log10()
        log_spec = torch.maximum(
            log_spec, log_spec.amax(dim=(1, 2), keepdim=True) - 8.0
        )
        return (log_spec + 4.0) / 4.0

In [ ]:
%%writefile turn_detector/augment.py
"""Waveform-level augmentations for turn detection.

The two augmentations that teach "silence != done":
  - trailing_silence: appended to ANY clip without changing its label — a
    finished speaker followed by silence is still finished, an unfinished one
    is still unfinished.
  - pause_cut: truncate a COMPLETE utterance at a speech-active point and
    append silence -> a genuine "paused mid-thought" example (label flips to
    incomplete). Applied on the fly in the dataset.

All functions take/return float32 mono @16kHz and are deterministic given rng.
"""

import numpy as np

SR = 16000


def trailing_silence(wav: np.ndarray, rng: np.random.Generator,
                     min_s: float = 0.2, max_s: float = 1.2) -> np.ndarray:
    n = int(rng.uniform(min_s, max_s) * SR)
    return np.concatenate([wav, np.zeros(n, dtype=np.float32)])


def _energy_envelope(wav: np.ndarray, frame: int = 400, hop: int = 160):
    n_frames = max(1, (len(wav) - frame) // hop + 1)
    idx = np.arange(n_frames)[:, None] * hop + np.arange(frame)[None, :]
    return np.sqrt((wav[idx.clip(max=len(wav) - 1)] ** 2).mean(axis=1)), hop


def pause_cut(wav: np.ndarray, rng: np.random.Generator,
              lo: float = 0.4, hi: float = 0.85,
              min_keep_s: float = 0.6, min_removed_s: float = 0.4):
    """Cut a complete utterance mid-speech. Returns wav or None if impossible."""
    env, hop = _energy_envelope(wav)
    thresh = max(env.max() * 0.15, 1e-4)
    active = np.nonzero(env > thresh)[0]
    if len(active) < 10:
        return None
    span_start, span_end = active[0], active[-1]
    span = span_end - span_start
    if span < 20:
        return None
    # candidate frames inside [lo, hi] of the active span that are speech-active
    frame_lo = span_start + int(span * lo)
    frame_hi = span_start + int(span * hi)
    candidates = active[(active >= frame_lo) & (active <= frame_hi)]
    if len(candidates) == 0:
        return None
    cut_frame = int(rng.choice(candidates))
    cut = cut_frame * hop
    if cut < min_keep_s * SR or len(wav) - cut < min_removed_s * SR:
        return None
    out = wav[:cut]
    return trailing_silence(out, rng)


def add_noise(wav: np.ndarray, rng: np.random.Generator,
              snr_lo: float = 10.0, snr_hi: float = 30.0) -> np.ndarray:
    rms = np.sqrt((wav ** 2).mean())
    if rms < 1e-5:
        return wav
    snr = rng.uniform(snr_lo, snr_hi)
    noise_rms = rms / (10 ** (snr / 20))
    return (wav + rng.normal(0, noise_rms, len(wav))).astype(np.float32)


def speed_perturb(wav: np.ndarray, rng: np.random.Generator,
                  lo: float = 0.9, hi: float = 1.1) -> np.ndarray:
    factor = rng.uniform(lo, hi)
    n_out = int(len(wav) / factor)
    x_old = np.arange(len(wav))
    x_new = np.linspace(0, len(wav) - 1, n_out)
    return np.interp(x_new, x_old, wav).astype(np.float32)

In [ ]:
%%writefile turn_detector/config.py
"""Experiment configurations E1-E4."""

import hashlib
import json
from dataclasses import asdict, dataclass, field


@dataclass
class ExperimentConfig:
    name: str
    arch: str = "whisper"              # whisper | tinymel
    use_hinglish_synth: bool = False

    # augmentation probabilities (train only)
    pause_cut_p: float = 0.0           # complete -> cut mid-speech, label flips to 0
    trailing_silence_p: float = 0.0
    noise_p: float = 0.0
    speed_p: float = 0.0

    # optimization
    epochs: int = 4
    batch_size: int = 64
    lr_encoder: float = 1e-5
    lr_head: float = 1e-4
    weight_decay: float = 0.01
    warmup_frac: float = 0.05
    grad_clip: float = 1.0
    seed: int = 42

    # bookkeeping
    checkpoint_every_steps: int = 500
    notes: str = ""

    def config_hash(self) -> str:
        return hashlib.sha1(
            json.dumps(asdict(self), sort_keys=True).encode()
        ).hexdigest()[:10]


EXPERIMENTS = {
    "e1_baseline": ExperimentConfig(
        name="e1_baseline",
        notes="EN+HI real data only, no augmentation, WhisperTinyTurn",
    ),
    "e2_hinglish_aug": ExperimentConfig(
        name="e2_hinglish_aug",
        use_hinglish_synth=True,
        pause_cut_p=0.15,
        trailing_silence_p=0.5,
        noise_p=0.25,
        speed_p=0.25,
        notes="headline model: +hinglish synth, +pause/silence/noise/speed aug",
    ),
    "e3_tinymel_scratch": ExperimentConfig(
        name="e3_tinymel_scratch",
        arch="tinymel",
        use_hinglish_synth=True,
        pause_cut_p=0.15,
        trailing_silence_p=0.5,
        noise_p=0.25,
        speed_p=0.25,
        epochs=8,
        lr_head=3e-4,
        notes="from-scratch ~1M param model, same data/aug as e2",
    ),
    "e4_no_pause_aug": ExperimentConfig(
        name="e4_no_pause_aug",
        use_hinglish_synth=True,
        pause_cut_p=0.0,
        trailing_silence_p=0.0,
        noise_p=0.25,
        speed_p=0.25,
        notes="ablation: e2 minus pause_cut and trailing_silence",
    ),
}

In [ ]:
%%writefile turn_detector/model.py
"""Turn-detection model architectures.

- WhisperTinyTurn: pretrained Whisper-Tiny encoder truncated to an 8s input
  window (positional embeddings sliced 1500 -> 400), + attention pooling +
  small MLP head. ~8M params.
- TinyMelNet: from-scratch comparison, ~1M params: depthwise-separable Conv1d
  stack over mel frames + BiGRU + the same pooling/head.

Both take log-mel (B, 80, 800) and return a single logit per example
(sigmoid -> P(turn complete)).
"""

import torch
import torch.nn as nn

from turn_detector.features import N_ENCODER_POSITIONS


class AttnPool(nn.Module):
    """Learned-query attention pooling over time: (B, T, D) -> (B, D)."""

    def __init__(self, dim: int):
        super().__init__()
        self.query = nn.Parameter(torch.randn(dim) * 0.02)
        self.scale = dim ** -0.5

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        weights = torch.softmax(x @ self.query * self.scale, dim=1)
        return (weights.unsqueeze(-1) * x).sum(dim=1)


def make_head(dim: int, hidden: int = 256, dropout: float = 0.1) -> nn.Module:
    return nn.Sequential(
        nn.LayerNorm(dim),
        nn.Linear(dim, hidden),
        nn.GELU(),
        nn.Dropout(dropout),
        nn.Linear(hidden, 1),
    )


def truncate_whisper_encoder(encoder, n_positions: int = N_ENCODER_POSITIONS):
    """Slice Whisper's 30s positional table to our window length, in place."""
    old = encoder.embed_positions
    new = nn.Embedding(n_positions, old.embedding_dim)
    new.weight.data.copy_(old.weight.data[:n_positions])
    encoder.embed_positions = new
    encoder.config.max_source_positions = n_positions
    if hasattr(encoder, "max_source_positions"):
        encoder.max_source_positions = n_positions
    return encoder


class WhisperTinyTurn(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        dim = encoder.config.d_model
        self.pool = AttnPool(dim)
        self.head = make_head(dim)

    @classmethod
    def from_pretrained(cls, name: str = "openai/whisper-tiny"):
        from transformers import WhisperModel
        encoder = WhisperModel.from_pretrained(name).encoder
        return cls(truncate_whisper_encoder(encoder))

    def forward(self, mel: torch.Tensor) -> torch.Tensor:
        hidden = self.encoder(mel).last_hidden_state       # (B, 400, 384)
        return self.head(self.pool(hidden)).squeeze(-1)


class DSConvBlock(nn.Module):
    """Depthwise-separable Conv1d + BN + GELU."""

    def __init__(self, channels: int, kernel: int = 5, stride: int = 1):
        super().__init__()
        self.depthwise = nn.Conv1d(
            channels, channels, kernel, stride=stride,
            padding=kernel // 2, groups=channels,
        )
        self.pointwise = nn.Conv1d(channels, channels, 1)
        self.norm = nn.BatchNorm1d(channels)
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(self.norm(self.pointwise(self.depthwise(x))))


class TinyMelNet(nn.Module):
    def __init__(self, width: int = 192, gru_hidden: int = 128):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(80, width, 5, stride=2, padding=2),
            nn.BatchNorm1d(width),
            nn.GELU(),
            DSConvBlock(width, stride=2),
            DSConvBlock(width, stride=2),
            DSConvBlock(width, stride=1),
        )                                                   # (B, width, 100)
        self.gru = nn.GRU(width, gru_hidden, batch_first=True, bidirectional=True)
        self.pool = AttnPool(2 * gru_hidden)
        self.head = make_head(2 * gru_hidden)

    def forward(self, mel: torch.Tensor) -> torch.Tensor:
        x = self.stem(mel).transpose(1, 2)                  # (B, 100, width)
        x, _ = self.gru(x)                                  # (B, 100, 2*hidden)
        return self.head(self.pool(x)).squeeze(-1)


def build_model(arch: str) -> nn.Module:
    if arch == "whisper":
        return WhisperTinyTurn.from_pretrained()
    if arch == "tinymel":
        return TinyMelNet()
    raise ValueError(f"unknown arch: {arch}")


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

In [ ]:
%%writefile turn_detector/dataset.py
"""Dataset over FLAC shards + parquet manifests.

Works identically on Kaggle (prep-notebook output + hinglish-synth dataset)
and locally (small subsets, unit tests). Multiple sources are concatenated;
each manifest row needs: id, path, label, language, split — optional:
midfiller, endfiller, synthetic, kind, source.

Augmentation policy (train split only) comes from ExperimentConfig. pause_cut
flips a complete example's label to incomplete on the fly; the sampler
compensates so the effective batch balance stays ~50/50.
"""

from pathlib import Path

import numpy as np
import polars as pl
import soundfile as sf
import torch
from torch.utils.data import Dataset, WeightedRandomSampler

from turn_detector import augment
from turn_detector.config import ExperimentConfig
from turn_detector.features import N_SAMPLES, right_align

OPTIONAL_COLS = {
    "midfiller": False, "endfiller": False, "synthetic": False,
    "kind": "", "source": "", "language": "",
}


def load_manifests(sources: list[tuple[str, str]], split: str) -> pl.DataFrame:
    """sources: [(manifest_parquet_path, audio_root), ...] -> unified frame."""
    frames = []
    for manifest_path, audio_root in sources:
        df = pl.read_parquet(manifest_path).filter(pl.col("split") == split)
        for col, default in OPTIONAL_COLS.items():
            if col not in df.columns:
                df = df.with_columns(pl.lit(default).alias(col))
            else:
                df = df.with_columns(pl.col(col).fill_null(default))
        df = df.with_columns(pl.lit(str(audio_root)).alias("audio_root"))
        frames.append(df.select(
            "id", "path", "label", "language", "split",
            "midfiller", "endfiller", "synthetic", "kind", "source", "audio_root",
        ))
    return pl.concat(frames)


class TurnDataset(Dataset):
    def __init__(self, manifest: pl.DataFrame, cfg: ExperimentConfig,
                 train: bool, seed_offset: int = 0):
        self.rows = manifest.to_dicts()
        self.cfg = cfg
        self.train = train
        self.seed_offset = seed_offset
        self.epoch = 0

    def set_epoch(self, epoch: int):
        self.epoch = epoch

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i: int):
        row = self.rows[i]
        wav, sr = sf.read(Path(row["audio_root"]) / row["path"], dtype="float32")
        if wav.ndim > 1:
            wav = wav.mean(axis=1)
        label = int(row["label"])

        if self.train:
            rng = np.random.default_rng(
                (self.cfg.seed + self.seed_offset) * 1_000_003
                + self.epoch * 101 + i
            )
            c = self.cfg
            if label == 1 and rng.random() < c.pause_cut_p:
                cut = augment.pause_cut(wav, rng)
                if cut is not None:
                    wav, label = cut, 0
            if rng.random() < c.trailing_silence_p:
                wav = augment.trailing_silence(wav, rng)
            if rng.random() < c.speed_p:
                wav = augment.speed_perturb(wav, rng)
            if rng.random() < c.noise_p:
                wav = augment.add_noise(wav, rng)

        wav = right_align(wav, N_SAMPLES)
        return torch.from_numpy(wav), torch.tensor(label, dtype=torch.float32), i

    def balanced_sampler(self, num_samples: int | None = None) -> WeightedRandomSampler:
        """50/50 sampler; completes oversampled to offset pause_cut label flips."""
        labels = np.array([r["label"] for r in self.rows])
        n_pos, n_neg = int(labels.sum()), int((1 - labels).sum())
        # fraction of drawn completes that stay complete after pause_cut
        keep = 1.0 - self.cfg.pause_cut_p if self.train else 1.0
        target_pos_draw = 0.5 / keep if keep > 0 else 0.5
        target_pos_draw = min(target_pos_draw, 0.75)
        w_pos = target_pos_draw / max(n_pos, 1)
        w_neg = (1 - target_pos_draw) / max(n_neg, 1)
        weights = np.where(labels == 1, w_pos, w_neg)
        return WeightedRandomSampler(
            torch.from_numpy(weights).double(),
            num_samples=num_samples or len(self.rows),
            replacement=True,
        )

In [ ]:
%%writefile turn_detector/train.py
"""Training, evaluation, and ONNX export for turn-detection experiments.

Step-based training (sampler draws with replacement, so batches are iid and a
mid-run resume just continues from the saved step — no epoch bookkeeping).
Checkpoints every cfg.checkpoint_every_steps to <out_dir>/ckpt_last.pt; a
killed Kaggle session loses at most that many steps. Resume is automatic when
the checkpoint's config hash matches.

Final artifacts in out_dir: ckpt_best.pt, model_fp32.onnx, model_int8.onnx,
metrics.json.

Note for security scanners: `model.eval()` below is PyTorch's inference-mode
switch, not code evaluation.
"""

import json
import math
import time
from pathlib import Path

import numpy as np
import polars as pl
import torch
from torch.utils.data import DataLoader

from turn_detector.config import ExperimentConfig
from turn_detector.dataset import TurnDataset, load_manifests
from turn_detector.features import LogMel
from turn_detector.model import build_model, count_params


# ---------------- metrics ----------------

def rank_auc(labels: np.ndarray, scores: np.ndarray) -> float:
    """ROC-AUC via the rank-sum statistic (no sklearn dependency)."""
    pos = scores[labels == 1]
    neg = scores[labels == 0]
    if len(pos) == 0 or len(neg) == 0:
        return float("nan")
    ranks = np.argsort(np.argsort(np.concatenate([pos, neg]))) + 1
    return float((ranks[: len(pos)].sum() - len(pos) * (len(pos) + 1) / 2)
                 / (len(pos) * len(neg)))


def f1_score(labels: np.ndarray, preds: np.ndarray) -> float:
    tp = int(((preds == 1) & (labels == 1)).sum())
    fp = int(((preds == 1) & (labels == 0)).sum())
    fn = int(((preds == 0) & (labels == 1)).sum())
    denom = 2 * tp + fp + fn
    return float(2 * tp / denom) if denom else float("nan")


def tune_threshold(labels: np.ndarray, probs: np.ndarray) -> float:
    best_t, best_acc = 0.5, 0.0
    for t in np.arange(0.05, 0.96, 0.01):
        acc = float(((probs >= t) == labels).mean())
        if acc > best_acc:
            best_acc, best_t = acc, float(t)
    return best_t


def slice_metrics(rows: pl.DataFrame, labels: np.ndarray, probs: np.ndarray,
                  threshold: float) -> dict:
    def compute(mask: np.ndarray) -> dict:
        if mask.sum() == 0:
            return {"n": 0}
        l, p = labels[mask], probs[mask]
        preds = (p >= threshold).astype(int)
        return {
            "n": int(mask.sum()),
            "acc_050": round(float(((p >= 0.5) == l).mean()), 4),
            "acc_tuned": round(float((preds == l).mean()), 4),
            "f1_tuned": round(f1_score(l, preds), 4),
            "auc": round(rank_auc(l, p), 4),
        }

    lang = rows["language"].to_numpy()
    midf = rows["midfiller"].fill_null(False).to_numpy().astype(bool)
    endf = rows["endfiller"].fill_null(False).to_numpy().astype(bool)
    synth = rows["synthetic"].fill_null(False).to_numpy().astype(bool)
    all_mask = np.ones(len(labels), dtype=bool)
    return {
        "overall": compute(all_mask),
        "english": compute(lang == "english"),
        "hindi": compute(lang == "hindi"),
        "hinglish": compute(lang == "hinglish"),
        "filler": compute(midf | endf),
        "human_audio": compute(~synth & (lang != "hinglish")),
        "threshold": threshold,
    }


# ---------------- eval ----------------

@torch.no_grad()
def predict(model, mel_fn, dataset, device, batch_size=64, num_workers=2):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=(device != "cpu"))
    model.eval()
    all_probs, all_labels = [], []
    for wav, label, _ in loader:
        mel = mel_fn(wav.to(device))
        with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
            logits = model(mel)
        all_probs.append(torch.sigmoid(logits.float()).cpu().numpy())
        all_labels.append(label.numpy())
    return np.concatenate(all_labels), np.concatenate(all_probs)


# ---------------- checkpointing ----------------

def save_ckpt(path: Path, model, opt, sched, scaler, step, best_val_auc, cfg_hash):
    torch.save({
        "model": model.state_dict(), "opt": opt.state_dict(),
        "sched": sched.state_dict(), "scaler": scaler.state_dict(),
        "step": step, "best_val_auc": best_val_auc, "cfg_hash": cfg_hash,
        "torch_rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }, path)


# ---------------- training ----------------

def train(cfg: ExperimentConfig, sources: dict, out_dir: str,
          device: str | None = None, steps_per_epoch: int | None = None,
          num_workers: int = 2):
    """sources: {"train": [(manifest, root), ...], "val": ..., "test": ...}"""
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(cfg.seed)
    np.random.seed(cfg.seed)

    train_df = load_manifests(sources["train"], "train")
    val_df = load_manifests(sources["val"], "val")
    print(f"train rows: {train_df.height}, val rows: {val_df.height}")
    train_ds = TurnDataset(train_df, cfg, train=True)
    val_ds = TurnDataset(val_df, cfg, train=False)

    spe = steps_per_epoch or math.ceil(train_df.height / cfg.batch_size)
    total_steps = spe * cfg.epochs
    warmup = max(1, int(total_steps * cfg.warmup_frac))

    model = build_model(cfg.arch).to(device)
    print(f"arch={cfg.arch} params={count_params(model):,}")
    mel_fn = LogMel().to(device)

    enc_params = [p for n, p in model.named_parameters() if n.startswith("encoder.")]
    other_params = [p for n, p in model.named_parameters() if not n.startswith("encoder.")]
    opt = torch.optim.AdamW(
        [{"params": enc_params, "lr": cfg.lr_encoder},
         {"params": other_params, "lr": cfg.lr_head}],
        weight_decay=cfg.weight_decay,
    )

    def lr_lambda(step):
        if step < warmup:
            return step / warmup
        p = (step - warmup) / max(1, total_steps - warmup)
        return 0.5 * (1 + math.cos(math.pi * p))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    scaler = torch.amp.GradScaler(enabled=(device == "cuda"))
    loss_fn = torch.nn.BCEWithLogitsLoss()

    # resume
    step, best_val_auc = 0, 0.0
    ckpt_last = out / "ckpt_last.pt"
    if ckpt_last.exists():
        state = torch.load(ckpt_last, map_location=device, weights_only=False)
        if state["cfg_hash"] == cfg.config_hash():
            model.load_state_dict(state["model"])
            opt.load_state_dict(state["opt"])
            sched.load_state_dict(state["sched"])
            scaler.load_state_dict(state["scaler"])
            step = state["step"]
            best_val_auc = state["best_val_auc"]
            print(f"resumed from step {step} (best val AUC {best_val_auc:.4f})")
        else:
            print("checkpoint config hash mismatch — starting fresh")

    history = []
    t_start = time.time()
    epoch_pass = step // spe
    while step < total_steps:
        train_ds.set_epoch(epoch_pass)
        loader = DataLoader(
            train_ds, batch_size=cfg.batch_size,
            sampler=train_ds.balanced_sampler(num_samples=spe * cfg.batch_size),
            num_workers=num_workers, pin_memory=(device == "cuda"),
            drop_last=True, persistent_workers=False,
        )
        model.train()
        for wav, label, _ in loader:
            if step >= total_steps:
                break
            mel = mel_fn(wav.to(device, non_blocking=True))
            label = label.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
                loss = loss_fn(model(mel), label)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(opt)
            scaler.update()
            sched.step()
            step += 1

            if step % 50 == 0:
                print(f"step {step}/{total_steps} loss {loss.item():.4f} "
                      f"lr {sched.get_last_lr()[-1]:.2e} "
                      f"({(time.time() - t_start) / 60:.1f} min)", flush=True)
            if step % cfg.checkpoint_every_steps == 0:
                save_ckpt(ckpt_last, model, opt, sched, scaler, step,
                          best_val_auc, cfg.config_hash())

            if step % spe == 0:  # epoch boundary -> validate
                vl, vp = predict(model, mel_fn, val_ds, device,
                                 cfg.batch_size, num_workers)
                val_auc = rank_auc(vl, vp)
                val_acc = float(((vp >= 0.5) == vl).mean())
                history.append({"step": step, "val_auc": round(val_auc, 4),
                                "val_acc_050": round(val_acc, 4)})
                print(f"  == step {step}: val AUC {val_auc:.4f} "
                      f"acc@0.5 {val_acc:.4f}", flush=True)
                if val_auc > best_val_auc:
                    best_val_auc = val_auc
                    torch.save({"model": model.state_dict(),
                                "cfg_hash": cfg.config_hash(),
                                "step": step, "val_auc": val_auc},
                               out / "ckpt_best.pt")
                save_ckpt(ckpt_last, model, opt, sched, scaler, step,
                          best_val_auc, cfg.config_hash())
                model.train()
        epoch_pass += 1

    # ---- final evaluation with best weights ----
    best = torch.load(out / "ckpt_best.pt", map_location=device, weights_only=False)
    model.load_state_dict(best["model"])

    vl, vp = predict(model, mel_fn, val_ds, device, cfg.batch_size, num_workers)
    threshold = tune_threshold(vl, vp)

    test_df = load_manifests(sources["test"], "test")
    test_ds = TurnDataset(test_df, cfg, train=False)
    tl, tp = predict(model, mel_fn, test_ds, device, cfg.batch_size, num_workers)
    test_metrics = slice_metrics(test_df, tl, tp, threshold)

    metrics = {
        "experiment": cfg.name,
        "config": cfg.__dict__,
        "params": count_params(model),
        "train_rows": train_df.height,
        "best_val_auc": round(best_val_auc, 4),
        "history": history,
        "threshold": threshold,
        "test": test_metrics,
        "train_minutes": round((time.time() - t_start) / 60, 1),
    }
    (out / "metrics.json").write_text(json.dumps(metrics, indent=2))
    print(json.dumps(test_metrics, indent=2))

    export_onnx(model, mel_fn, out, cfg, test_df, test_ds, threshold, metrics)
    return metrics


# ---------------- export ----------------

def export_onnx(model, mel_fn, out: Path, cfg, test_df, test_ds,
                threshold, metrics):
    import onnxruntime as ort
    model = model.cpu().eval()
    dummy = torch.randn(1, 80, 800)
    fp32_path = out / "model_fp32.onnx"
    # static batch=1: turn detection inference is streaming, one window at a
    # time, and dynamic batch breaks the bidirectional-GRU reshape on export
    try:  # legacy exporter: consistent shape metadata, quantizer-friendly
        torch.onnx.export(
            model, (dummy,), str(fp32_path),
            input_names=["mel"], output_names=["logit"],
            opset_version=17, dynamo=False,
        )
    except TypeError:  # older torch without the dynamo kwarg
        torch.onnx.export(
            model, (dummy,), str(fp32_path),
            input_names=["mel"], output_names=["logit"], opset_version=17,
        )

    # torch vs onnx parity (per-sample, batch=1 graph)
    sess = ort.InferenceSession(str(fp32_path), providers=["CPUExecutionProvider"])
    diffs = []
    for _ in range(4):
        x = torch.randn(1, 80, 800)
        with torch.no_grad():
            ref = torch.sigmoid(model(x)).numpy()
        got = 1 / (1 + np.exp(-sess.run(None, {"mel": x.numpy()})[0]))
        diffs.append(np.abs(ref - got).max())
    parity = float(max(diffs))
    print(f"onnx fp32 parity: max |dprob| = {parity:.2e}")

    from onnxruntime.quantization import QuantType, quantize_dynamic
    int8_path = out / "model_int8.onnx"
    quantize_dynamic(str(fp32_path), str(int8_path), weight_type=QuantType.QUInt8)

    # int8 accuracy on a stratified test subset (bounded runtime on CPU)
    rng = np.random.default_rng(0)
    labels_np = test_df["label"].to_numpy()
    idx = np.concatenate([
        rng.permutation(np.nonzero(labels_np == c)[0])[:1000] for c in (0, 1)
    ])
    sess8 = ort.InferenceSession(str(int8_path), providers=["CPUExecutionProvider"])
    probs, labels = [], []
    mel_cpu = mel_fn.cpu()
    for i in idx:
        wav, label, _ = test_ds[int(i)]
        mel = mel_cpu(wav.unsqueeze(0)).numpy()
        logit = sess8.run(None, {"mel": mel})[0][0]
        probs.append(1 / (1 + np.exp(-logit)))
        labels.append(int(label))
    labels, probs = np.array(labels), np.array(probs).ravel()
    int8_metrics = {
        "n": len(labels),
        "acc_tuned": round(float(((probs >= threshold) == labels).mean()), 4),
        "auc": round(rank_auc(labels, probs), 4),
        "size_mb": round(int8_path.stat().st_size / 1e6, 2),
        "fp32_size_mb": round(fp32_path.stat().st_size / 1e6, 2),
        "fp32_parity_max_dprob": parity,
    }
    metrics["int8_subset"] = int8_metrics
    (out / "metrics.json").write_text(json.dumps(metrics, indent=2))
    print("int8:", json.dumps(int8_metrics, indent=2))

In [ ]:
EXPERIMENT = "e1_baseline"   # e1_baseline | e2_hinglish_aug | e3_tinymel_scratch | e4_no_pause_aug
PREP = "/kaggle/input/smart-turn-enhi-prep/prep"
HINGLISH = "/kaggle/input/hinglish-synth"
RESUME_FROM = ""             # e.g. "/kaggle/input/02-train/run_e2_hinglish_aug"

In [ ]:
import shutil, sys
from pathlib import Path

sys.path.insert(0, ".")
from turn_detector.config import EXPERIMENTS
from turn_detector.train import train

cfg = EXPERIMENTS[EXPERIMENT]
out_dir = Path("/kaggle/working") / f"run_{EXPERIMENT}"
out_dir.mkdir(parents=True, exist_ok=True)

if RESUME_FROM and Path(RESUME_FROM, "ckpt_last.pt").exists():
    for f in Path(RESUME_FROM).glob("*"):
        if not (out_dir / f.name).exists():
            shutil.copy(f, out_dir / f.name)
    print(f"copied previous run from {RESUME_FROM}")

real = [(f"{PREP}/manifest.parquet", PREP)]
synth = [(f"{HINGLISH}/manifest.parquet", HINGLISH)]
train_sources = real + (synth if cfg.use_hinglish_synth else [])
sources = {
    "train": train_sources,
    "val": train_sources,
    "test": real + synth,   # always evaluate hinglish slice, even for e1
}

metrics = train(cfg, sources, str(out_dir), num_workers=3)

In [ ]:
import json
from pathlib import Path

m = json.loads((Path("/kaggle/working") / f"run_{EXPERIMENT}" / "metrics.json").read_text())
print(json.dumps({k: m[k] for k in ("experiment", "params", "best_val_auc",
                                    "threshold", "train_minutes")}, indent=2))
print(json.dumps(m["test"], indent=2))
print(json.dumps(m.get("int8_subset", {}), indent=2))
print("\nNow: Save Version, then download run_" + EXPERIMENT + "/ into the repo's experiments/ folder.")